# pyaegean neural-lemmatizer spike — multi-task GreBERTa

Shared `bowphs/GreBerta` encoder + **9 heads**: lemma edit-tree (the target) + UPOS + 7 morph dimensions (case/number/gender/tense/mood/voice/person). The aux heads are trained jointly so the encoder learns the morphology that *determines* the lemma — the lever for **unseen** forms. Prior runs: 53.4% (AGDT-only) → 58.2% (multi-treebank) unseen; target > stanza's 62.8%.

**Loop:** GPU runtime (H100 ideal) → Run all → cell 6b prints `DEV lemma all/UNSEEN` → download `spike_model.zip`. torch/transformers used only here; inference is onnxruntime-only.

In [ ]:
!nvidia-smi -L  # MUST list a GPU
%pip -q install 'transformers>=4.46' 'datasets>=2.19' 'accelerate>=0.30' onnx onnxruntime onnxscript

## 0 · Detect GPU + precision

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, then reconnect.'
USE_BF16 = torch.cuda.is_bf16_supported()
BS = 64 if USE_BF16 else 16
print(f'torch {torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}')
print(f'bf16={USE_BF16} -> batch_size={BS}')

## 1 · Upload `spike_data.zip` + load the label vocabularies

In [ ]:
import json, zipfile, pathlib
from google.colab import files
up = files.upload()  # pick spike_data.zip
zipfile.ZipFile(next(n for n in up if n.endswith('.zip'))).extractall('.')
DATA = pathlib.Path('data') if pathlib.Path('data/labels.json').exists() else pathlib.Path('.')
lj = json.loads((DATA / 'labels.json').read_text(encoding='utf-8'))
labels = lj['trees']                       # lemma edit-trees (head 0; used by the eval)
DIMS = ['upos','case','number','gender','tense','mood','voice','person']
FIELDS = ['lemma'] + DIMS                  # head/output order; lemma is 0
head_sizes = [len(labels)] + [len(lj[d]) for d in DIMS]
print('heads:', dict(zip(FIELDS, head_sizes)))

## 2 · Load training data

In [ ]:
from datasets import Dataset
rows = [json.loads(l) for l in open(DATA / 'train.jsonl', encoding='utf-8')]
ds = Dataset.from_list(rows)
print(ds)

## 3 · Tokenizer + tokenize, aligning ALL 9 label fields to first sub-tokens

In [ ]:
from transformers import AutoTokenizer
MODEL = 'bowphs/GreBerta'
tokenizer = AutoTokenizer.from_pretrained(MODEL, add_prefix_space=True)
def align(batch):
    enc = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True, max_length=512)
    out_labels = []
    for i in range(len(batch['tokens'])):
        word_ids = enc.word_ids(i); prev = None; rows_ = []
        for wid in word_ids:
            if wid is None or wid == prev:
                rows_.append([-100] * len(FIELDS))
            else:
                rows_.append([batch[f][i][wid] for f in FIELDS])
            prev = wid
        out_labels.append(rows_)
    enc['labels'] = out_labels
    return enc
tok_ds = ds.map(align, batched=True, remove_columns=ds.column_names)
tok_ds = tok_ds.train_test_split(test_size=0.02, seed=0)
print(tok_ds)

## 4 · Multi-head model (shared encoder + one linear head per field)

In [ ]:
import torch.nn as nn
from transformers import AutoModel
class MultiHead(nn.Module):
    def __init__(self, name, sizes):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleList([nn.Linear(h, n) for n in sizes])
    def forward(self, input_ids, attention_mask=None):
        x = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x = self.dropout(x)
        return tuple(head(x) for head in self.heads)
model = MultiHead(MODEL, head_sizes)

## 5 · Fine-tune (multi-task loss = sum of per-head cross-entropy; 12 epochs, best by val loss)
Watch `eval_loss` fall and plateau. The real lemma number is cell 6b.

In [ ]:
import torch
from transformers import TrainingArguments, Trainer
def collate(features):
    labs = [f['labels'] for f in features]
    batch = tokenizer.pad({'input_ids': [f['input_ids'] for f in features],
                           'attention_mask': [f['attention_mask'] for f in features]},
                          return_tensors='pt')
    maxlen = batch['input_ids'].size(1); H = len(labs[0][0])
    batch['labels'] = torch.tensor([la + [[-100]*H]*(maxlen-len(la)) for la in labs], dtype=torch.long)
    return batch
AUX_W = 0.1  # lemma (head 0) at full weight; POS/morph heads are auxiliary regularizers
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        logits = model(inputs['input_ids'], inputs.get('attention_mask'))
        ce = [nn.functional.cross_entropy(lg.reshape(-1, lg.size(-1)),
              labels[..., h].reshape(-1), ignore_index=-100) for h, lg in enumerate(logits)]
        loss = ce[0] + AUX_W * sum(ce[1:])
        return (loss, logits) if return_outputs else loss
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        with torch.no_grad():
            loss = self.compute_loss(model, dict(inputs))
        return (loss.detach(), None, None)
args = TrainingArguments(
    output_dir='out', learning_rate=5e-5, lr_scheduler_type='cosine',
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS*2,
    num_train_epochs=12, weight_decay=0.01, warmup_ratio=0.06,
    bf16=USE_BF16, fp16=not USE_BF16, tf32=USE_BF16, optim='adamw_torch_fused',
    dataloader_num_workers=2, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    remove_unused_columns=False, label_names=['labels'], logging_steps=50, report_to='none')
trainer = MultiTaskTrainer(model=model, args=args, train_dataset=tok_ds['train'],
                           eval_dataset=tok_ds['test'], data_collator=collate)
trainer.train()

## 6 · Export to ONNX (fp32) — manual export of the multi-output model
Outputs named per field, **`lemma` first**, so the eval reads output 0. Ship fp32 (int8 per-tensor quant collapses the ~10k-class head; production = per-channel + validation).

In [ ]:
import os
os.makedirs('onnx', exist_ok=True)
m = model.eval().float().cpu()
dummy = (torch.tensor([[0, 1, 2, 3, 4]]), torch.ones(1, 5, dtype=torch.long))
dyn = {'input_ids': {0: 'b', 1: 's'}, 'attention_mask': {0: 'b', 1: 's'}}
dyn.update({f: {0: 'b', 1: 's'} for f in FIELDS})
try:  # torch>=2.9 defaults to the dynamo exporter (needs onnxscript); legacy is simpler here
    torch.onnx.export(m, dummy, 'onnx/model.onnx',
        input_names=['input_ids', 'attention_mask'], output_names=FIELDS,
        dynamic_axes=dyn, opset_version=14, dynamo=False)
    print('exported (legacy)')
except Exception as e:
    print('legacy failed:', str(e)[:150], '\n-> retrying with the dynamo exporter...')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxscript'])
    torch.onnx.export(m, dummy, 'onnx/model.onnx',
        input_names=['input_ids', 'attention_mask'], output_names=FIELDS,
        dynamic_axes=dyn, opset_version=14)
    print('exported (dynamo)')
tokenizer.save_pretrained('onnx')
print('exported', os.path.getsize('onnx/model.onnx') // (1024*1024), 'MB; outputs:', FIELDS)

## 6b · In-Colab dev lemma accuracy (reads lemma head = output 0)
The number that matters — see it before downloading.

In [ ]:
import numpy as np, onnxruntime as ort, re, unicodedata
from tokenizers import Tokenizer
sess = ort.InferenceSession('onnx/model.onnx', providers=['CPUExecutionProvider'])
tkf = Tokenizer.from_file('onnx/tokenizer.json')
inames = {i.name for i in sess.get_inputs()}
def _nrm(s): return unicodedata.normalize('NFC', s)
def _clean(s): return re.sub(r'\d+$', '', unicodedata.normalize('NFC', s))
def apply_tree(t, w):
    if t[0] == 'keep': return w
    if t[0] == 'sub': return t[1]
    _, pl, sl, lf, rt = t
    if pl + sl > len(w): return None
    p = apply_tree(lf, w[:pl]); s = apply_tree(rt, w[len(w)-sl:] if sl else '')
    if p is None or s is None: return None
    return p + (w[pl:len(w)-sl] if sl else w[pl:]) + s
dev = [json.loads(l) for l in open(DATA / 'dev.jsonl', encoding='utf-8')]
na = nu = oa = ou = 0
for srow in dev:
    forms = srow['tokens']; enc = tkf.encode(forms, is_pretokenized=True)
    feed = {'input_ids': np.array([enc.ids], dtype=np.int64)}
    if 'attention_mask' in inames: feed['attention_mask'] = np.array([enc.attention_mask], dtype=np.int64)
    lg = sess.run(None, feed)[0][0]          # output 0 = lemma logits
    first = {}
    for pos, wid in enumerate(enc.word_ids):
        if wid is not None and wid not in first: first[wid] = pos
    for wi, form in enumerate(forms):
        if not srow['scored'][wi]: continue
        na += 1; un = not srow['seen'][wi]; nu += un
        pos = first.get(wi); pred = _nrm(form)
        if pos is not None:
            out = apply_tree(json.loads(labels[int(np.argmax(lg[pos]))]), _nrm(form))
            if out is not None: pred = out
        if _clean(pred) == srow['lemmas'][wi]: oa += 1; ou += un
print(f'DEV lemma — all {oa/na:.1%}  UNSEEN {ou/nu:.1%}   (beat: stanza 62.8% unseen; pure-Python 40.3%)')

## 7 · Package + download

In [ ]:
import shutil
pathlib.Path('ship').mkdir(exist_ok=True)
shutil.copy('onnx/model.onnx', 'ship/model.onnx')
shutil.copy('onnx/tokenizer.json', 'ship/tokenizer.json')
shutil.copy(str(DATA / 'labels.json'), 'ship/labels.json')
shutil.make_archive('spike_model', 'zip', 'ship')
files.download('spike_model.zip')

## Next
Report cell 6b's `DEV lemma` (unseen > 62.8% = we beat stanza). Send `spike_model.zip` and I confirm locally with `eval_spike.py` (reads output 0 = lemma, unchanged).